# Clustering Analysis for Facial Emotion Recognition

This notebook implements comprehensive clustering analysis using geometric features extracted from facial images.

## Configuration

**To change which emotions are analyzed**, edit the `SELECTED_EMOTIONS` variable in the configuration cell below.

Examples:
- `SELECTED_EMOTIONS = ['Happy', 'Sad']` - Analyze only Happy and Sad
- `SELECTED_EMOTIONS = ['Happy', 'Sad', 'Surprised']` - Analyze three emotions
- `SELECTED_EMOTIONS = None` - Analyze all emotions

## Methods Implemented:
1. **K-Means Clustering** with Elbow Method
2. **Hierarchical Clustering** (Agglomerative) with Ward linkage
3. **DBSCAN** for outlier detection
4. **Visualization** of results
5. **Internal and External Validation**

## Questions to Answer:
- Does the Elbow happen at K equal to the number of emotions or a different K?
- How well do the selected emotions separate into distinct clusters?
- Does DBSCAN find distinct clusters for each emotion or treat some as outliers?


In [ ]:
# Imports and Configuration
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Tuple, Dict, List
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import (
    silhouette_score, 
    calinski_harabasz_score, 
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_score,
    completeness_score,
    v_measure_score
)
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

from preprocess4 import load_and_prepare_data, DEFAULT_OUTPUT_FILE
from visualize_clusters import visualize_clusters_with_images, compare_clusters_vs_true_labels

# ============================================================================
# CONFIGURATION: Change this to select which emotions to analyze
# ============================================================================
SELECTED_EMOTIONS = ['Happy', 'Sad', 'Surprised', 'Anger','Contempt','Disgust','Fear','Neutral']  # Change this list to include different emotions
# Examples:
# SELECTED_EMOTIONS = ['Happy', 'Sad', 'Surprised']  # 3 emotions
# SELECTED_EMOTIONS = ['Anger', 'Disgust']  # 2 emotions
# SELECTED_EMOTIONS = None  # All emotions (use None for all)

# Clustering parameters (automatically adjusted based on number of emotions)
NUM_EMOTIONS = len(SELECTED_EMOTIONS) if SELECTED_EMOTIONS else 8  # Default to 8 if all emotions
K_RANGE_START = 2
K_RANGE_END = max(10, NUM_EMOTIONS + 3)  # Test up to NUM_EMOTIONS + 3

# Output directory
_BASE_DIR = Path.cwd() if Path.cwd().name == 'four' else Path.cwd() / 'four'
OUTPUT_DIR = _BASE_DIR / 'clustering_results'
OUTPUT_DIR.mkdir(exist_ok=True)

# Dataset folder for aligned images
DATASET_FOLDER = _BASE_DIR / 'facial_emotion_recognition_aligned'

print("=" * 80)
print("CONFIGURATION")
print("=" * 80)
print(f"Selected emotions: {SELECTED_EMOTIONS if SELECTED_EMOTIONS else 'ALL'}")
print(f"Number of emotions: {NUM_EMOTIONS}")
print(f"K range: {K_RANGE_START} to {K_RANGE_END-1}")
print(f"Output directory: {OUTPUT_DIR}")
print("=" * 80)
print("All imports successful!")


## Helper Functions


In [ ]:
def kmeans_elbow_method(X: np.ndarray, k_range: range = range(2, 15), random_state: int = 42):
    """Perform K-Means clustering with Elbow Method to find optimal K."""
    inertias = {}
    kmeans_models = {}
    
    print("Running K-Means Elbow Method...")
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        kmeans.fit(X)
        inertias[k] = kmeans.inertia_
        kmeans_models[k] = kmeans
        print(f"  K={k}: Inertia={kmeans.inertia_:.2f}")
    
    # Find elbow point
    k_list = list(k_range)
    inertia_list = [inertias[k] for k in k_list]
    deltas = np.diff(inertia_list)
    deltas2 = np.diff(deltas)
    
    if len(deltas2) > 0:
        elbow_idx = np.argmax(np.abs(deltas2)) + 1
        optimal_k = k_list[min(elbow_idx, len(k_list) - 1)]
    else:
        optimal_k = k_list[len(k_list) // 2]
    
    return inertias, optimal_k, kmeans_models


def plot_elbow_method(inertias: Dict[int, float], optimal_k: int, num_emotions: int):
    """Plot the Elbow Method graph."""
    k_values = sorted(inertias.keys())
    inertia_values = [inertias[k] for k in k_values]
    
    plt.figure(figsize=(10, 6))
    plt.plot(k_values, inertia_values, 'bo-', linewidth=2, markersize=8)
    plt.axvline(x=optimal_k, color='r', linestyle='--', linewidth=2, label=f'Suggested K={optimal_k}')
    plt.axvline(x=num_emotions, color='g', linestyle='--', linewidth=2, 
                label=f'K={num_emotions} (Number of emotions)')
    if num_emotions + 1 in k_values:
        plt.axvline(x=num_emotions + 1, color='orange', linestyle='--', linewidth=2, 
                    label=f'K={num_emotions + 1}')
    plt.xlabel('Number of Clusters (K)', fontsize=12)
    plt.ylabel('Inertia (Within-cluster sum of squares)', fontsize=12)
    plt.title('K-Means Elbow Method', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def hierarchical_clustering(X: np.ndarray, n_clusters: int = None, linkage_method: str = 'ward'):
    """Perform Hierarchical Clustering (Agglomerative) with Ward linkage."""
    print(f"Running Hierarchical Clustering with {linkage_method} linkage...")
    
    if n_clusters is None:
        n_clusters = 7
    
    model = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage_method)
    labels = model.fit_predict(X)
    print(f"  Created {n_clusters} clusters")
    return model, labels


def plot_dendrogram(X: np.ndarray, linkage_method: str = 'ward', max_display: int = 50, 
                    num_emotions: int = None):
    """Plot dendrogram for hierarchical clustering."""
    print(f"Creating dendrogram with {linkage_method} linkage...")
    
    if len(X) > max_display:
        indices = np.random.choice(len(X), max_display, replace=False)
        X_subset = X[indices]
        print(f"  Using subset of {max_display} samples for dendrogram visualization")
    else:
        X_subset = X
        indices = np.arange(len(X))
    
    linkage_matrix = linkage(X_subset, method=linkage_method)
    
    plt.figure(figsize=(15, 8))
    dendrogram(
        linkage_matrix,
        truncate_mode='level',
        p=10,
        leaf_rotation=90,
        leaf_font_size=8,
        show_contracted=True
    )
    plt.title(f'Hierarchical Clustering Dendrogram ({linkage_method} linkage)', 
              fontsize=14, fontweight='bold')
    plt.xlabel('Sample Index or (Cluster Size)', fontsize=12)
    plt.ylabel('Distance', fontsize=12)
    
    if num_emotions and len(linkage_matrix) >= num_emotions:
        plt.axhline(y=linkage_matrix[-num_emotions, 2], color='r', linestyle='--', 
                    label=f'Cut at K={num_emotions} (number of emotions)')
    if num_emotions and len(linkage_matrix) >= num_emotions + 1:
        plt.axhline(y=linkage_matrix[-(num_emotions + 1), 2], color='g', linestyle='--', 
                    label=f'Cut at K={num_emotions + 1}')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    return linkage_matrix


def dbscan_clustering(X: np.ndarray, eps: float = 0.5, min_samples: int = 5):
    """Perform DBSCAN clustering for outlier detection."""
    print(f"Running DBSCAN (eps={eps}, min_samples={min_samples})...")
    
    model = DBSCAN(eps=eps, min_samples=min_samples)
    labels = model.fit_predict(X)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    print(f"  Found {n_clusters} clusters")
    print(f"  Found {n_noise} outliers (noise points)")
    
    return model, labels


def estimate_eps_from_kdistance(X: np.ndarray, k: int = 4, percentile: int = 90):
    """
    Estimate eps parameter using k-distance graph method.
    This helps find the 'elbow' in the k-distance plot.
    """
    from sklearn.neighbors import NearestNeighbors
    
    # Calculate k-nearest neighbors distances
    neighbors = NearestNeighbors(n_neighbors=k)
    neighbors_fit = neighbors.fit(X)
    distances, indices = neighbors_fit.kneighbors(X)
    
    # Get k-th nearest neighbor distance for each point
    k_distances = distances[:, k-1]
    k_distances_sorted = np.sort(k_distances)[::-1]
    
    # Use percentile as estimate (often the "elbow" is around 90th percentile)
    estimated_eps = np.percentile(k_distances, percentile)
    
    return estimated_eps, k_distances_sorted


def plot_kdistance_graph(k_distances_sorted: np.ndarray, suggested_eps: float):
    """Plot k-distance graph to help choose eps parameter."""
    plt.figure(figsize=(10, 6))
    plt.plot(range(len(k_distances_sorted)), k_distances_sorted, 'b-', linewidth=2)
    plt.axhline(y=suggested_eps, color='r', linestyle='--', linewidth=2, 
                label=f'Suggested eps={suggested_eps:.3f} (90th percentile)')
    plt.xlabel('Points sorted by distance', fontsize=12)
    plt.ylabel(f'k-distance (k=4)', fontsize=12)
    plt.title('K-Distance Graph (for DBSCAN eps estimation)', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def tune_dbscan(X: np.ndarray, eps_range: List[float], min_samples_range: List[int]):
    """Tune DBSCAN parameters by testing different combinations."""
    print("Tuning DBSCAN parameters...")
    results = []
    
    for eps in eps_range:
        for min_samples in min_samples_range:
            try:
                # Run DBSCAN directly (without calling dbscan_clustering to avoid print spam)
                from sklearn.cluster import DBSCAN
                model = DBSCAN(eps=eps, min_samples=min_samples)
                labels = model.fit_predict(X)
                
                n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
                n_noise = list(labels).count(-1)
                
                if n_clusters >= 2:
                    try:
                        sil_score = silhouette_score(X, labels)
                    except:
                        sil_score = -1
                else:
                    sil_score = -1
                
                results.append({
                    'eps': eps,
                    'min_samples': min_samples,
                    'n_clusters': n_clusters,
                    'n_noise': n_noise,
                    'silhouette_score': sil_score
                })
            except Exception as e:
                print(f"Error with eps={eps}, min_samples={min_samples}: {e}")
                results.append({
                    'eps': eps,
                    'min_samples': min_samples,
                    'n_clusters': 0,
                    'n_noise': len(X),
                    'silhouette_score': -1
                })
    
    df = pd.DataFrame(results)
    print(f"Completed tuning: {len(df)} parameter combinations tested")
    return df


def visualize_clusters_2d(X: np.ndarray, labels: np.ndarray, true_labels: np.ndarray, 
                          method_name: str, title_suffix: str = ""):
    """Visualize clusters in 2D using PCA for dimensionality reduction."""
    print(f"Creating 2D visualization for {method_name}...")
    
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Clusters
    unique_labels_cluster = sorted(set(labels))
    colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels_cluster)))
    
    for i, label in enumerate(unique_labels_cluster):
        if label == -1:
            mask = labels == label
            ax1.scatter(X_2d[mask, 0], X_2d[mask, 1], 
                       c='black', marker='x', s=50, alpha=0.5, label='Outliers')
        else:
            mask = labels == label
            ax1.scatter(X_2d[mask, 0], X_2d[mask, 1], 
                       c=[colors[i]], label=f'Cluster {label}', alpha=0.6, s=50)
    
    ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
    ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
    ax1.set_title(f'{method_name} - Clusters{title_suffix}', fontsize=12, fontweight='bold')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: True labels
    unique_true = sorted(set(true_labels))
    colors_true = plt.cm.Set3(np.linspace(0, 1, len(unique_true)))
    
    for i, label in enumerate(unique_true):
        mask = true_labels == label
        ax2.scatter(X_2d[mask, 0], X_2d[mask, 1], 
                   c=[colors_true[i]], label=label, alpha=0.6, s=50)
    
    ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
    ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
    ax2.set_title(f'{method_name} - True Labels{title_suffix}', fontsize=12, fontweight='bold')
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def internal_validation(X: np.ndarray, labels: np.ndarray, method_name: str) -> Dict[str, float]:
    """Perform Internal Validation (The 'Math' Score)."""
    print(f"\nInternal Validation for {method_name}:")
    
    unique_labels = set(labels)
    if len(unique_labels) < 2 or (len(unique_labels) == 2 and -1 in unique_labels):
        print("  Warning: Not enough clusters for validation")
        return {'silhouette_score': np.nan, 'calinski_harabasz_score': np.nan, 
                'davies_bouldin_score': np.nan}
    
    mask = labels != -1
    if mask.sum() < 2:
        print("  Warning: Too many outliers")
        return {'silhouette_score': np.nan, 'calinski_harabasz_score': np.nan, 
                'davies_bouldin_score': np.nan}
    
    X_filtered = X[mask]
    labels_filtered = labels[mask]
    
    metrics = {}
    
    try:
        metrics['silhouette_score'] = silhouette_score(X_filtered, labels_filtered)
        print(f"  Silhouette Score: {metrics['silhouette_score']:.4f} (higher is better, range: -1 to 1)")
    except:
        metrics['silhouette_score'] = np.nan
        print("  Silhouette Score: Could not compute")
    
    try:
        metrics['calinski_harabasz_score'] = calinski_harabasz_score(X_filtered, labels_filtered)
        print(f"  Calinski-Harabasz Index: {metrics['calinski_harabasz_score']:.4f} (higher is better)")
    except:
        metrics['calinski_harabasz_score'] = np.nan
        print("  Calinski-Harabasz Index: Could not compute")
    
    try:
        metrics['davies_bouldin_score'] = davies_bouldin_score(X_filtered, labels_filtered)
        print(f"  Davies-Bouldin Index: {metrics['davies_bouldin_score']:.4f} (lower is better)")
    except:
        metrics['davies_bouldin_score'] = np.nan
        print("  Davies-Bouldin Index: Could not compute")
    
    return metrics


def external_validation(labels: np.ndarray, true_labels: np.ndarray, method_name: str) -> Dict[str, float]:
    """Perform External Validation (The 'Truth' Score)."""
    print(f"\nExternal Validation for {method_name}:")
    
    mask = labels != -1
    if mask.sum() == 0:
        print("  Warning: All points are outliers")
        return {'adjusted_rand_score': np.nan, 'normalized_mutual_info_score': np.nan,
                'homogeneity_score': np.nan, 'completeness_score': np.nan, 'v_measure_score': np.nan}
    
    labels_filtered = labels[mask]
    true_labels_filtered = true_labels[mask]
    
    metrics = {}
    
    try:
        metrics['adjusted_rand_score'] = adjusted_rand_score(true_labels_filtered, labels_filtered)
        print(f"  Adjusted Rand Index: {metrics['adjusted_rand_score']:.4f} (higher is better, range: -1 to 1)")
    except:
        metrics['adjusted_rand_score'] = np.nan
        print("  Adjusted Rand Index: Could not compute")
    
    try:
        metrics['normalized_mutual_info_score'] = normalized_mutual_info_score(
            true_labels_filtered, labels_filtered
        )
        print(f"  Normalized Mutual Information: {metrics['normalized_mutual_info_score']:.4f} (higher is better, range: 0 to 1)")
    except:
        metrics['normalized_mutual_info_score'] = np.nan
        print("  Normalized Mutual Information: Could not compute")
    
    try:
        metrics['homogeneity_score'] = homogeneity_score(true_labels_filtered, labels_filtered)
        metrics['completeness_score'] = completeness_score(true_labels_filtered, labels_filtered)
        metrics['v_measure_score'] = v_measure_score(true_labels_filtered, labels_filtered)
        print(f"  Homogeneity: {metrics['homogeneity_score']:.4f} (higher is better)")
        print(f"  Completeness: {metrics['completeness_score']:.4f} (higher is better)")
        print(f"  V-measure: {metrics['v_measure_score']:.4f} (higher is better)")
    except:
        metrics['homogeneity_score'] = np.nan
        metrics['completeness_score'] = np.nan
        metrics['v_measure_score'] = np.nan
        print("  Homogeneity/Completeness/V-measure: Could not compute")
    
    return metrics

print("Helper functions defined!")


## Load Data


In [ ]:
# Load data using selected emotions from configuration
print("=" * 80)
print("LOADING DATA")
print("=" * 80)
if SELECTED_EMOTIONS:
    print(f"Loading data for emotions: {SELECTED_EMOTIONS}")
else:
    print("Loading data for ALL emotions")

# Use aligned dataset CSV
CSV_PATH = _BASE_DIR / 'facial_features_aligned.csv'
print(f"Using CSV: {CSV_PATH}")

# Check if CSV has Image_Path column (if not, will be reconstructed from Person_ID/Image_Name)
df_check = pd.read_csv(CSV_PATH, nrows=1)
if 'Image_Path' not in df_check.columns:
    print("\nNote: CSV doesn't have Image_Path column.")
    print("Image paths will be reconstructed from Person_ID and Image_Name.")
    print("To add Image_Path column, re-run preprocess_dataset() in preprocess4.py")

X_scaled, y_labels, scaler, selected_indices = load_and_prepare_data(
    csv_path=CSV_PATH,
    emotions=SELECTED_EMOTIONS,  # Use configuration variable
    num_randoms=0
)

# Store selected_indices for use in visualization
print(f"\nSelected indices range: {selected_indices.min()} to {selected_indices.max()}")
print(f"These indices will be used to match images with cluster labels.")

print(f"\nDataset shape: {X_scaled.shape}")
print(f"Number of samples: {len(X_scaled)}")
print(f"Number of features: {X_scaled.shape[1]}")
print(f"Emotions: {sorted(set(y_labels))}")

# Convert labels to numeric for some metrics
le = LabelEncoder()
y_numeric = le.fit_transform(y_labels)

results_summary = {}


## 1. K-Means Clustering with Elbow Method

**Question:** Does the Elbow happen at K equal to the number of emotions or a different K?


In [ ]:
print("=" * 80)
print("1. K-MEANS CLUSTERING WITH ELBOW METHOD")
print("=" * 80)

# Use K range from configuration
inertias, optimal_k, kmeans_models = kmeans_elbow_method(
    X_scaled, 
    k_range=range(K_RANGE_START, K_RANGE_END)
)

# Plot elbow method
plot_elbow_method(inertias, optimal_k, NUM_EMOTIONS)

# Answer the question
print(f"\nQuestion: Does the Elbow happen at K={NUM_EMOTIONS} (number of emotions) or a different K?")
print(f"  Suggested optimal K (by algorithm): {optimal_k}")
if NUM_EMOTIONS in inertias:
    print(f"  K={NUM_EMOTIONS} inertia: {inertias[NUM_EMOTIONS]:.2f}")
if NUM_EMOTIONS - 1 in inertias:
    print(f"  K={NUM_EMOTIONS - 1} inertia: {inertias[NUM_EMOTIONS - 1]:.2f}")
if NUM_EMOTIONS + 1 in inertias:
    print(f"  K={NUM_EMOTIONS + 1} inertia: {inertias[NUM_EMOTIONS + 1]:.2f}")


In [ ]:
# Run K-Means with K equal to number of emotions
k_optimal = NUM_EMOTIONS
print(f"\nRunning K-Means with K={k_optimal} (number of emotions)...")
kmeans_optimal = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
kmeans_labels_optimal = kmeans_optimal.fit_predict(X_scaled)

# Visualize
visualize_clusters_2d(X_scaled, kmeans_labels_optimal, y_labels, f'K-Means (K={k_optimal})')

# Validation
internal_metrics_optimal = internal_validation(X_scaled, kmeans_labels_optimal, f'K-Means (K={k_optimal})')
external_metrics_optimal = external_validation(kmeans_labels_optimal, y_labels, f'K-Means (K={k_optimal})')

results_summary[f'K-Means (K={k_optimal})'] = {
    'internal': internal_metrics_optimal,
    'external': external_metrics_optimal
}

# Visualize images by cluster
print("\n" + "=" * 80)
print("VISUALIZING IMAGES BY CLUSTER")
print("=" * 80)
visualize_clusters_with_images(
    kmeans_labels_optimal, 
    y_labels,
    csv_path=CSV_PATH,
    dataset_folder=DATASET_FOLDER,
    method_name=f'K-Means (K={k_optimal})',
    max_images_per_cluster=None,  # Show all images in each cluster
    selected_indices=selected_indices  # Pass the exact indices used
)


In [ ]:
# Run K-Means with K+1 (for comparison - over-clustering)
k_comparison = NUM_EMOTIONS + 1
if k_comparison <= K_RANGE_END - 1:
    print(f"\nRunning K-Means with K={k_comparison} (for comparison - over-clustering)...")
    kmeans_comparison = KMeans(n_clusters=k_comparison, random_state=42, n_init=10)
    kmeans_labels_comparison = kmeans_comparison.fit_predict(X_scaled)
    
    # Visualize
    visualize_clusters_2d(X_scaled, kmeans_labels_comparison, y_labels, f'K-Means (K={k_comparison})')
    
    # Validation
    internal_metrics_comparison = internal_validation(X_scaled, kmeans_labels_comparison, f'K-Means (K={k_comparison})')
    external_metrics_comparison = external_validation(kmeans_labels_comparison, y_labels, f'K-Means (K={k_comparison})')
    
    results_summary[f'K-Means (K={k_comparison})'] = {
        'internal': internal_metrics_comparison,
        'external': external_metrics_comparison
    }


## 2. Hierarchical Clustering (Agglomerative)

**Observation:** How well do the selected emotions separate? Are they clearly distinct clusters?


In [ ]:
print("=" * 80)
print("2. HIERARCHICAL CLUSTERING (AGGLOMERATIVE)")
print("=" * 80)

# Plot dendrogram
linkage_matrix = plot_dendrogram(X_scaled, linkage_method='ward', num_emotions=NUM_EMOTIONS)

# Analyze emotion similarity
print("\nAnalyzing emotion similarity in hierarchical clustering...")
print(f"\n  Emotion distribution:")
unique_emotions = sorted(set(y_labels))
for emotion in unique_emotions:
    emotion_mask = np.array([label.lower() == emotion.lower() for label in y_labels])
    print(f"  {emotion} samples: {emotion_mask.sum()}")
print("\n  Check the dendrogram above to see how the emotions cluster!")


In [ ]:
# Run Hierarchical Clustering with K equal to number of emotions
k_optimal = NUM_EMOTIONS
print(f"\nRunning Hierarchical Clustering with K={k_optimal} (number of emotions)...")
hier_model_optimal, hier_labels_optimal = hierarchical_clustering(
    X_scaled, n_clusters=k_optimal, linkage_method='ward'
)

# Visualize
visualize_clusters_2d(X_scaled, hier_labels_optimal, y_labels, f'Hierarchical (K={k_optimal})')

# Validation
internal_metrics_hier_optimal = internal_validation(
    X_scaled, hier_labels_optimal, f'Hierarchical (K={k_optimal})'
)
external_metrics_hier_optimal = external_validation(
    hier_labels_optimal, y_labels, f'Hierarchical (K={k_optimal})'
)

results_summary[f'Hierarchical (K={k_optimal})'] = {
    'internal': internal_metrics_hier_optimal,
    'external': external_metrics_hier_optimal
}

# Visualize images by cluster
print("\n" + "=" * 80)
print("VISUALIZING IMAGES BY CLUSTER")
print("=" * 80)
visualize_clusters_with_images(
    hier_labels_optimal, 
    y_labels,
    csv_path=CSV_PATH,
    dataset_folder=DATASET_FOLDER,
    method_name=f'Hierarchical (K={k_optimal})',
    max_images_per_cluster=None,  # Show all images in each cluster
    selected_indices=selected_indices  # Pass the exact indices used
)


In [ ]:
# Run Hierarchical Clustering with K+1 (for comparison - over-clustering)
k_comparison = NUM_EMOTIONS + 1
if k_comparison <= K_RANGE_END - 1:
    print(f"\nRunning Hierarchical Clustering with K={k_comparison} (for comparison)...")
    hier_model_comparison, hier_labels_comparison = hierarchical_clustering(
        X_scaled, n_clusters=k_comparison, linkage_method='ward'
    )
    
    # Visualize
    visualize_clusters_2d(X_scaled, hier_labels_comparison, y_labels, f'Hierarchical (K={k_comparison})')
    
    # Validation
    internal_metrics_hier_comparison = internal_validation(
        X_scaled, hier_labels_comparison, f'Hierarchical (K={k_comparison})'
    )
    external_metrics_hier_comparison = external_validation(
        hier_labels_comparison, y_labels, f'Hierarchical (K={k_comparison})'
    )
    
    results_summary[f'Hierarchical (K={k_comparison})'] = {
        'internal': internal_metrics_hier_comparison,
        'external': external_metrics_hier_comparison
    }


## 3. DBSCAN Clustering (Outlier Detection)

**Hypothesis:** DBSCAN will find distinct clusters for the selected emotions, or treat some samples as outliers if they are geometrically similar.


In [ ]:
print("=" * 80)
print("3. DBSCAN CLUSTERING (OUTLIER DETECTION)")
print("=" * 80)

# First, estimate eps using k-distance method
print("\nEstimating eps parameter using k-distance graph...")
estimated_eps, k_distances = estimate_eps_from_kdistance(X_scaled, k=4)
print(f"Estimated eps (90th percentile): {estimated_eps:.3f}")

# Plot k-distance graph
plot_kdistance_graph(k_distances, estimated_eps)

# Create eps range around the estimated value
# Use wider range to find good parameters
eps_min = max(0.1, estimated_eps * 0.3)
eps_max = estimated_eps * 2.5
eps_range = np.linspace(eps_min, eps_max, 8).tolist()
eps_range = [round(e, 2) for e in eps_range]

# Adjust min_samples based on dataset size
# Rule of thumb: min_samples should be at least the number of dimensions
# But also consider dataset size - don't make it too large
min_samples_min = max(2, X_scaled.shape[1])  # At least number of features
min_samples_max = min(10, len(X_scaled) // 10)  # At most 10% of dataset
min_samples_range = list(range(min_samples_min, min_samples_max + 1, 1))

print(f"\nTuning parameters:")
print(f"  eps range: {eps_range}")
print(f"  min_samples range: {min_samples_range}")

tuning_results = tune_dbscan(X_scaled, eps_range, min_samples_range)
print("\nDBSCAN Tuning Results:")
print(f"Number of parameter combinations tested: {len(tuning_results)}")
print(f"Columns: {list(tuning_results.columns) if not tuning_results.empty else 'No results'}")
display(tuning_results)

# Check if we got results
if tuning_results.empty:
    print("\nWARNING: No tuning results returned! Check the tune_dbscan function.")
else:
    # Save tuning results
    tuning_results.to_csv(OUTPUT_DIR / 'dbscan_tuning_results.csv', index=False)
    print(f"\nSaved DBSCAN tuning results to {OUTPUT_DIR / 'dbscan_tuning_results.csv'}")


In [ ]:
# Find best parameters
# Check if tuning_results is empty or missing columns
if tuning_results.empty:
    print("\nWarning: No tuning results found. Using default parameters.")
    best_eps = estimated_eps
    best_min_samples = max(2, X_scaled.shape[1])
elif 'n_clusters' not in tuning_results.columns:
    print("\nWarning: Tuning results missing expected columns. Using default parameters.")
    print(f"Available columns: {list(tuning_results.columns)}")
    best_eps = estimated_eps
    best_min_samples = max(2, X_scaled.shape[1])
else:
    # Prefer parameters that create reasonable number of clusters (not all outliers, not all one cluster)
    valid_results = tuning_results[
        (tuning_results['n_clusters'] >= 1) & 
        (tuning_results['n_clusters'] <= NUM_EMOTIONS + 2) &
        (tuning_results['n_noise'] < len(X_scaled) * 0.8)  # Less than 80% outliers
    ]
    
    if len(valid_results) > 0:
        # Among valid results, prefer higher silhouette score
        best_params = valid_results.loc[valid_results['silhouette_score'].idxmax()]
        print(f"\nFound {len(valid_results)} valid parameter combinations")
        best_eps = best_params['eps']
        best_min_samples = int(best_params['min_samples'])
        print(f"\nBest DBSCAN parameters:")
        print(f"  eps: {best_eps}")
        print(f"  min_samples: {best_min_samples}")
        print(f"  Expected clusters: {int(best_params['n_clusters'])}")
        print(f"  Expected outliers: {int(best_params['n_noise'])}")
    else:
        # If no valid results, use the one with least outliers
        print("\nWarning: No ideal parameters found. Using parameters with least outliers.")
        best_params = tuning_results.loc[tuning_results['n_noise'].idxmin()]
        best_eps = best_params['eps']
        best_min_samples = int(best_params['min_samples'])
        print(f"\nBest DBSCAN parameters (fallback):")
        print(f"  eps: {best_eps}")
        print(f"  min_samples: {best_min_samples}")
        print(f"  Expected clusters: {int(best_params['n_clusters'])}")
        print(f"  Expected outliers: {int(best_params['n_noise'])}")

# Run DBSCAN with best parameters
dbscan_model, dbscan_labels = dbscan_clustering(X_scaled, eps=best_eps, min_samples=best_min_samples)

# Visualize
visualize_clusters_2d(
    X_scaled, dbscan_labels, y_labels,
    f'DBSCAN (eps={best_eps}, min_samples={best_min_samples})'
)

# Visualize images by cluster (if not all outliers)
if len(set(dbscan_labels)) > 1 or -1 not in dbscan_labels:
    print("\n" + "=" * 80)
    print("VISUALIZING IMAGES BY CLUSTER")
    print("=" * 80)
    visualize_clusters_with_images(
        dbscan_labels, 
        y_labels,
        csv_path=CSV_PATH,
        dataset_folder=DATASET_FOLDER,
        method_name=f'DBSCAN (eps={best_eps}, min_samples={best_min_samples})',
        max_images_per_cluster=None,  # Show all images in each cluster
        selected_indices=selected_indices  # Pass the exact indices used
    )
else:
    print("\nSkipping image visualization: All points are outliers.")

# Validation
internal_metrics_dbscan = internal_validation(
    X_scaled, dbscan_labels, 
    f'DBSCAN (eps={best_eps}, min_samples={best_min_samples})'
)
external_metrics_dbscan = external_validation(dbscan_labels, y_labels, 'DBSCAN')

results_summary['DBSCAN'] = {
    'internal': internal_metrics_dbscan,
    'external': external_metrics_dbscan
}


## Summary of Results


In [ ]:
# Create summary DataFrame
summary_data = []
for method, metrics in results_summary.items():
    row = {'Method': method}
    row.update({f'Internal_{k}': v for k, v in metrics['internal'].items()})
    row.update({f'External_{k}': v for k, v in metrics['external'].items()})
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print("Summary of Results:")
display(summary_df)

# Save summary
summary_df.to_csv(OUTPUT_DIR / 'clustering_results_summary.csv', index=False)
print(f"\nSaved summary to {OUTPUT_DIR / 'clustering_results_summary.csv'}")


### Internal Validation (Math Scores)

These metrics evaluate clustering quality without using true labels.


In [ ]:
print("=" * 80)
print("INTERNAL VALIDATION (Math Scores)")
print("=" * 80)

for method, metrics in results_summary.items():
    print(f"\n{method}:")
    for metric, value in metrics['internal'].items():
        if not np.isnan(value):
            print(f"  {metric}: {value:.4f}")


### External Validation (Truth Scores)

These metrics compare cluster labels to true emotion labels.


In [ ]:
print("=" * 80)
print("EXTERNAL VALIDATION (Truth Scores)")
print("=" * 80)

for method, metrics in results_summary.items():
    print(f"\n{method}:")
    for metric, value in metrics['external'].items():
        if not np.isnan(value):
            print(f"  {metric}: {value:.4f}")


## Analysis Complete!

All results have been saved to the `clustering_results` directory.
